In [222]:
keyToNep = {
  "a": "\u093E",# ा
  "b": "\u092C",# ब
  "c": "\u091B",# छ
  "d": "\u0926",# द
  "e": "\u0947",# े
  "f": "\u0909",# उ
  "g": "\u0917",# ग
  "h": "\u0939",# ह
  "i": "\u093F",# ि
  "j": "\u091C",# ज
  "k": "\u0915",# क
  "l": "\u0932",# ल
  "m": "\u092E",# म
  "n": "\u0928",# न
  "o": "\u094B",# ो
  "p": "\u092A",# प
  "q": "\u091F",# ट
  "r": "\u0930",# र
  "s": "\u0938",# स
  "t": "\u0924",# त
  "u": "\u0941",# ु
  "v": "\u0935",# व
  "w": "\u094C",# ौ
  "x": "\u0921",# ड
  "y": "\u092F",# य
  "z": "\u0937",# ष
  "A": "\u0906",# आ
  "B": "\u092D",# भ
  "C": "\u091A",# च
  "D": "\u0927",# ध
  "E": "\u0948",# ै
  "F": "\u090A",# ऊ
  "G": "\u0918",# घ
  "H": "\u0905",# अ
  "I": "\u0940",# ी
  "J": "\u091D",# झ
  "K": "\u0916",# ख
  "L": "\u0933",# ळ
  "M": "\u0902",# ं
  "N": "\u0923",# ण
  "O": "\u0913",# ओ
  "P": "\u092B",# फ
  "Q": "\u0920",# ठ
  "R": "\u0943",# ृ
  "S": "\u0936",# श
  "T": "\u0925",# थ
  "U": "\u0942",# ू
  "V": "\u0901",# ँ
  "W": "\u0914",# औ
  "X": "\u0922",# ढ
  "Y": "\u091E",# ञ
  "Z": "\u090B",# ऋ
  "0": "\u0966",# ०
  "1": "\u0967",# १
  "2": "\u0968",# २
  "3": "\u0969",# ३
  "4": "\u096A",# ४
  "5": "\u096B",# ५
  "6": "\u096C",# ६
  "7": "\u096D",# ७
  "8": "\u096E",# ८
  "9": "\u096F",# ९
  "^": "\u005E",# ^
  "`": "\u093D",# ऽ
  "~": "\u093C",# ़
  "_": "\u0952",# ॒
  "+": "\u200C",# 
  "=": "\u200D",#
  "[": "\u0907",# इ
  "{": "\u0908",# ई
  "]": "\u090F",# ए
  "}": "\u0910", # ऐ
  "\\": "\u0950",#/ ॐ
  "|": "\u0903",# ः
  "<": "\u0919",# ङ
  ".": "\u0964",# ।
  ">": "\u0965",# ॥
  "/": "\u094D",# ्
  "?": "\u003F",# ?
}

def formatKey(key):
  return keyToNep[key]

def roman2nepali(roman: str):
  return "".join([formatKey(char) for char in roman])


In [223]:
import pandas as pd
import numpy as np


all = pd.read_csv("words.csv")
common = pd.read_csv("common_words_in_poem.csv")
all.shape

(113393, 1)

In [233]:
letters = [
    "ब", "छ", "द", "उ", "ग", "ह", "ज", "क", "ल", "म", "न", "प", 
    "ट", "र", "स", "त", "व", "ड", "य", "ष", "आ", "भ", "च", "ध", "ऊ", 
    "घ", "अ", "झ", "ख", "ळ", "ण", "ओ", "फ", "ठ", "श", "थ", 
    "औ", "ढ", "ञ", "ऋ", "इ", "ई", "ए", "ऐ", "ङ", "ऑ"
]

extensions = [
    "ा", "ि", "ी", "ु", "ू", "ृ", "े", "ै", "ो", "ौ",  # vowel signs
    "ं", "ँ", "ः",  # anusvara, chandrabindu, visarga
    "़", "ऽ", "्", "॒"  # nukta, avagraha, halant, anudatta
]

symbols = [
    "^", "`", "~", "_", "+", "=", "[", "{", "]", "}", "\\", "|", "<", ".", ">", "/", "?", "!", "'", '"', "।", ",","‘","“"
]

excluding = list("00123456789")


def get_unique_words(
    file: str = None,
    txt: str = None,
    letters=letters,
    extensions=extensions,
    symbols=symbols,
    excluding=excluding,
):
    words = []

    corpus = ""
    if txt:
        corpus += txt
    if file:
        with open(file, 'r') as f:
            corpus+=f.read()

    if not txt and not file:
        return words

    corpuslist = corpus.split()
    for token in corpuslist:
        if token.strip() in symbols+excluding:
            continue
        else:
            token = token.strip()
            word1 = ""
            for c in token:
                if c in symbols+excluding:
                    continue
                else:
                    word1+=c

            words.append(word1)

    words = pd.Series(common_words_in_poems).value_counts().index.to_numpy()

    return words


common_words_in_poems = get_unique_words('lspd.txt')

In [225]:
all['words'].head()

0        अ
1       अ:
2       अँ
3    अँएरो
4    अँकरा
Name: words, dtype: object

In [226]:
from rhyming_detector import rhyme_score
def get_topk_rhyming_words(word: str, k: int=10):
    rhyming_words = []
    for wd in common.words:
        score = rhyme_score(word,wd,mode="loose",strict_alignment=True)
        if score>3.5:
            rhyming_words.append({wd: score })

    rhyming_words.sort(key=lambda x: list(x.values())[0], reverse=True)

    words = [list(d.keys())[0] for d in rhyming_words][1:]

    if len(words) < k or k is None:
        return words
    else:
        return words[0:k]

In [227]:
import fasttext
import random

ft = fasttext.load_model('cc.ne.300.bin')

In [228]:
def similarity_score(e1: list | np.ndarray, e2: list | np.ndarray) -> float:
    """
    Cosine similarity between two vectors.
    Handles 1D or 2D inputs (e.g., column/row vectors) by flattening to 1D.
    """
    v1 = np.asarray(e1, dtype=np.float64).ravel()   # flatten to 1D
    v2 = np.asarray(e2, dtype=np.float64).ravel()

    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)

    if norm1 == 0.0 or norm2 == 0.0:
        return 0.0

    return float(dot / (norm1 * norm2))

def get_similarity_score(ctx, word, vecfinder = ft.get_word_vector):
    lines = ctx.splitlines()
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return 0.0
    vectors = [ft.get_sentence_vector(line) for line in lines]
    vctx = np.mean(vectors, axis=0)
    vword = vecfinder(word)
    return similarity_score(vctx,vword)


ctx = """
भाका, भूल, दया, क्षमा र ममता, सन्तोष जान्दैन त्यो,
इन्द्रै बिन्ति गरुन् झुकेर पदमा त्यो बिन्ति मान्दैन त्यो,
थुप्रोमा उधिनी मिठो र नमिठो रोजेर छान्दैन त्यो,
खाता जाँची सबै दुरुस्त नबुझी बिर्सेर हान्दैन त्यो ।१।

राजा रङ्क सबै समान उसका वैषम्य गर्दैन त्यो,
आयो टप्प टिप्यो, लग्यो, मिति पुग्यो टारेर र्टर्दैन त्यो,
लाखौँ औषध अस्त्रशस्त्र महिमा देखेर र्डर्दैन त्यो,
व्याधातुल्य लुकेर चल्दछ सदा मारेर मर्दैन त्यो ।२।

आँसुका दहमा नुहाउँछ चिसो पानी रुचाउन्न त्यो,
सुख्खा जर्जर अस्थिपञ्जर विना शैया बनाउन्न त्यो,
मैलो भष्मसिवाय अङ्गभरमा केही लगाउन्न त्यो,
हाहाकार सरी मिठो अरु कुनै संङ्गीत गाउन्न त्यो ।३।

जोजो मिल्छ सुलुक्क निल्छ मुखमा हाली चपाउन्न त्यो
थाल्यो च्वाम्म सबै चपाउन भने आहार पाउन्न त्यो,
जत्ती मिल्छ उती उकेल्दछ पनि केही पचाउन्न त्यो,
यै चालासित कल्पकल्प कहिल्यै खाई अघाउन्न त्यो ।४। 
"""

word = random.choice(ctx.split())
print(word)
get_similarity_score(ctx, word)

भूल,


-0.013155829412287143

In [229]:
import uuid
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client import models

# client = QdrantClient(url="http://localhost:6333")
client = QdrantClient(":memory:")
COLLECTION_NAME = "fasttext_words"

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=300, distance=Distance.COSINE),
    )

In [ ]:
def get_paragraph_vector(paragraph: str):
    lines = paragraph.splitlines()
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return np.zeros(300).tolist()
    
    vectors = [ft.get_sentence_vector(line) for line in lines]
    vctx = np.mean(vectors, axis=0)
    
    return vctx.tolist()


class qdrantAPI:
    @staticmethod
    def store_vector(word: str, vector: list):
        """
        Stores a word and its vector inside Qdrant.
        """
        point_id = str(uuid.uuid4())
        
        client.upsert(
            collection_name=COLLECTION_NAME,
            points=[
                PointStruct(
                    id=point_id,
                    vector=vector if isinstance(vector, list) else vector.tolist(),
                    payload={"word": word}
                )
            ]
        )

    @staticmethod
    def store_word(word: str):
        word_vector = ft.get_word_vector(word)
        qdrantAPI.store_vector(word, word_vector)

    @staticmethod
    def get_topk_words(ctx: str, k: int = 10):
        """
        ctx is a multiline paragraph.
        Retrieves the top k most similar words from Qdrant.
        """
        vctx = get_paragraph_vector(ctx)
        
        # Query Qdrant using the paragraph vector
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=vctx,
            limit=k,
            with_payload=True
        ).points
        
        top_k_words = [point.payload["word"] for point in search_result]
        return top_k_words

    @staticmethod
    def get_word_vector(word: str):
        word_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key="word",
                    match=models.MatchValue(value=word)
                )
            ]
        )
        
        result = client.query_points(
            collection_name=COLLECTION_NAME,
            query_filter=word_filter,
            limit=1,
            with_vectors=True
        ).points
        
        if result:
            return result[0].vector
        return None

In [231]:
for word in common.words:
    qdrantAPI.store_word(word)

In [274]:
w = random.choice(ctx.split())
print(w)
rwords = get_topk_rhyming_words(w, 10)
ctxvec = get_paragraph_vector(ctx)
edict = {}
for word in rwords:
    score =get_similarity_score(ctx, word)
    edict[word] = score


print(rwords)
print(list(dict(sorted(edict.items(), key=lambda x: x[1], reverse=True)).keys()))

त्यो,
['द्रोण', 'स्रोत', 'क्रोध', 'द्रोह', 'श्लोक', 'बिछोड', 'वियोग', 'यूतोक', 'बिजोग', 'निचोर']
['द्रोह', 'निचोर', 'बिजोग', 'क्रोध', 'बिछोड', 'वियोग', 'श्लोक', 'स्रोत', 'द्रोण', 'यूतोक']
